In [17]:
import os
import sys
from dotenv import load_dotenv
from typing import Any
from langchain_community.docstore.document import Document
from langchain_openai import ChatOpenAI
from langchain_core.retrievers import BaseRetriever
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import RetrievalQA
from sentence_transformers import CrossEncoder
from pydantic import Field,BaseModel
from utils.evaluate_rag import *
from utils.helper_functions import *

load_dotenv("/Users/nilasark/advanced/.env")
path="Understanding_Climate_Change.pdf"

In [5]:
vectorestore=encode_pdf(path)

In [10]:
class RatingScore(BaseModel):
    relevance_score:float=Field(...,description="The relevance score of a document to a query")
def rerank_documents(query:str,docs:list[Document],top_k:int=4):
    prompt_template=PromptTemplate(
    input_variables=["query", "doc"],
    template="""On a scale of 1-10, rate the relevance of the following document to the query. Consider the specific context and intent of the query, not just keyword matches.
        Query: {query}
        Document: {doc}
        Relevance Score:"""
        )

    llm=ChatOpenAI(model='gpt-4o-mini',temperature=0,max_completion_tokens=4000)
    llm_chain=prompt_template|llm.with_structured_output(schema=RatingScore)
    scored_docs=[]
    for doc in docs:
        content={"query":query,"doc":doc.page_content}
        ranked_result=llm_chain.invoke(content).relevance_score
        try:
            score=float(ranked_result)
        except ValueError:
            score=0
        scored_docs.append((doc,score))

    reranked_docs=sorted(scored_docs,key=lambda x:x[1],reverse=True)
    return [doc for doc,_ in reranked_docs[:top_k]]


In [14]:
query="What are the impacts of climate change on biodiversity?"
initial_docs=vectorestore.similarity_search(query,k=3)
reranked_docs=rerank_documents(query,initial_docs,4)

print(f"Top Initial Documents:")
for i,doc in enumerate(initial_docs[:3]):
    print(f"Doc_{i}:{doc.page_content[:200]}...")

print(f"\nQuery:{query}")
print("\n")
for i,doc in enumerate(reranked_docs[:3]):
    print(f"Doc_{i}:{doc.page_content[:200]}...")

Top Initial Documents:
Doc_0:Impact on Ecosystems 
Terrestrial Ecosystems 
Climate change is altering terrestrial ecosystems by shifting habitat ranges, changing species 
distributions, and impacting ecosystem functions. Forests,...
Doc_1:goals. Policies should promote synergies between biodiversity conservation and climate 
action. 
Chapter 10: Climate Change and Human Health 
Health Impacts 
Heat-Related Illnesses 
Rising temperature...
Doc_2:such as Fridays for Future, demonstrate the power of young voices in advocating for a 
sustainable future. 
By continuing to innovate, collaborate, and commit to sustainable practices, we can mitigate...

Query:What are the impacts of climate change on biodiversity?


Doc_0:Impact on Ecosystems 
Terrestrial Ecosystems 
Climate change is altering terrestrial ecosystems by shifting habitat ranges, changing species 
distributions, and impacting ecosystem functions. Forests,...
Doc_1:such as Fridays for Future, demonstrate the power of young voices 

In [25]:
from pydantic import ConfigDict
class CustomRetriever(BaseRetriever,BaseModel):
    vectorestore:Any=Field(description="Vector Store for initial retirval")
    model_config=ConfigDict(arbitrary_types_allowed=True)

    def _get_relevant_documents(self,query:str,top_n:int=3)->list[Document]:
        initial_docs=vectorestore.similarity_search(query=query,k=30)
        return rerank_documents(query,initial_docs,top_k=top_n)
    async def _aget_relevant_documents(self, query:str)->list[Document]:
        return self._get_relevant_docs(query=query)

custom_retriever=CustomRetriever(vectorestore=vectorestore)

llm=ChatOpenAI(model='gpt-4o-mini',temperature=0)

qa_chain=RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=custom_retriever,
    return_source_documents=True
)

In [26]:
result=qa_chain({'query':query})
print(f"\nQuery:{query}")
print(f"Answer: {result['result']}")

for i,docs in enumerate(result['source_documents']):
    print(f"\nDoc{i}:{docs.page_content[:200]}....")


Query:What are the impacts of climate change on biodiversity?
Answer: Climate change impacts biodiversity by altering terrestrial and marine ecosystems. In terrestrial ecosystems, it shifts habitat ranges, changes species distributions, and affects ecosystem functions, leading to a loss of biodiversity and disruption of ecological balance. In marine ecosystems, rising sea temperatures, ocean acidification, and changing currents affect marine biodiversity, causing species migration and changes in reproductive cycles that can disrupt marine food webs and fisheries. Overall, these changes threaten the stability and health of various ecosystems.

Doc0:Impact on Ecosystems 
Terrestrial Ecosystems 
Climate change is altering terrestrial ecosystems by shifting habitat ranges, changing species 
distributions, and impacting ecosystem functions. Forests,....

Doc1:such as Fridays for Future, demonstrate the power of young voices in advocating for a 
sustainable future. 
By continuing to innovat

In [27]:
from sentence_transformers import CrossEncoder

model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L6-v2')
scores = model.predict([
    ("How many people live in Berlin?", "Berlin had a population of 3,520,031 registered inhabitants in an area of 891.82 square kilometers."),
    ("How many people live in Berlin?", "Berlin is well known for its museums."),
])
print(scores)
# [ 8.607138 -4.320078]

[ 8.607141 -4.320078]


In [32]:
from pydantic import ConfigDict
cross_encoder=CrossEncoder('cross-encoder/ms-marco-MiniLM-L6-v2')
class CrossEncoderRetriever(BaseRetriever,BaseModel):
    vectorstore:Any=Field(description="VectorStore for Initial Retrieval")
    cross_encoder:Any=Field(description="Cross Encoder For reranking")
    top_k:Any=Field(default=5,description="Initial Docs to be retrieved")
    top_n:Any=Field(default=3,description="Docs to be sent as result")

    model_config=ConfigDict(arbitrary_types_allowed=True)

    def _get_relevant_documents(self,query:str)->list[Document]:
        initial_results=vectorestore.similarity_search(query=query,k=self.top_k)
        pairs=[(query,docs.page_content)for docs in initial_results]
        rerank_scores=self.cross_encoder.predict(pairs)
        scored_docs=sorted(zip(initial_docs,rerank_scores),key=lambda x:x[1],reverse=True)
        return [docs for docs,_ in scored_docs[:self.top_n]]

    def _aget_relevant_documents(self,query:str)->list[Document]:
        return self._get_relevant_documents(query)


In [36]:
cross_encoder_retriever=CrossEncoderRetriever(
    vectorstore=vectorestore,
    cross_encoder=cross_encoder,
    top_k=10,
    top_n=5
)
llm=ChatOpenAI(model='gpt-4o-mini',temperature=0)
qa_chain=RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=cross_encoder_retriever,
    return_source_documents=True
)
query = "What are the impacts of climate change on biodiversity?"
result=qa_chain({'query':query})

print(f"\nQuery:{query}")
print(f"Answer:{result['result']}\n")
for i,docs in enumerate(result['source_documents']):
    print(f"DOCUMENT {i}")
    print('='*100)
    print(f"{docs.page_content}")
    print(f"{'='*100}\n")





Query:What are the impacts of climate change on biodiversity?
Answer:Climate change impacts biodiversity by altering terrestrial and marine ecosystems. In terrestrial ecosystems, it shifts habitat ranges, changes species distributions, and affects ecosystem functions, leading to a loss of biodiversity and disruption of ecological balance. In marine ecosystems, rising sea temperatures, ocean acidification, and changing currents affect marine biodiversity, causing species migration and changes in reproductive cycles that can disrupt marine food webs and fisheries.

DOCUMENT 0
Impact on Ecosystems 
Terrestrial Ecosystems 
Climate change is altering terrestrial ecosystems by shifting habitat ranges, changing species 
distributions, and impacting ecosystem functions. Forests, grasslands, and deserts are 
experiencing shifts in plant and animal species composition. These changes can lead to a loss 
of biodiversity and disrupt ecological balance. 
Marine Ecosystems 
Marine ecosystems are hig